In [20]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import seaborn as sns
import numpy as np

datasets = [
    {
        "participant_tsv": "/nfs/trident3/lightsheet/prado/mouse_app_lecanemab_batch2/bids/participants.tsv",
        "spimquant_dir": "/nfs/trident3/lightsheet/prado/mouse_app_lecanemab_batch2/derivatives/spimquant-v0.6.0rc2_84a605e_ozx",
    },
    {
        "participant_tsv": "/nfs/trident3/lightsheet/prado/mouse_app_lecanemab_batch3/bids/participants.tsv",
        "spimquant_dir": "/nfs/trident3/lightsheet/prado/mouse_app_lecanemab_batch3/derivatives/spimquant-v0.6.0rc2_84a605e_ozx",
    },
]

In [21]:
def load_subject_df(spimquant_dir: str, subject: str) -> pd.DataFrame | None:
    
    regionpropstats_tsv = (
        f"{spimquant_dir}/{subject}/micr/"
        f"{subject}_sample-brain_acq-imaris4x_stain-Abeta_seg-all_from-ABAv3_level-5_desc-otsu+k3i2_regionpropstats.tsv"
    )

    if not Path(regionpropstats_tsv).exists():
        return None

    
    df_subject = pd.read_csv(regionpropstats_tsv,sep="\t")
    df_subject["subject"] = subject
    return df_subject

In [22]:
dataset_dfs = []

for dataset in datasets:
    df_participants = pd.read_csv(dataset["participant_tsv"], sep="\t")

    subject_dfs = [
        df_subject
        for subject in df_participants["participant_id"]
        if (df_subject := load_subject_df(dataset["spimquant_dir"], subject)) is not None
    ]

    if not subject_dfs:
        continue

    

    df_dataset = pd.concat(subject_dfs, ignore_index=False).merge(
        df_participants,
        left_on="subject",
        right_on="participant_id",
        how="left",
    )

    dataset_dfs.append(df_dataset)

df = pd.concat(dataset_dfs, ignore_index=True)

In [23]:
#keep only lecanemab and PBS (vehicle)
df = df.query("treatment == 'Lecanemab' or treatment == 'PBS'")

In [24]:
df

,pos_x,pos_y,pos_z,nvoxels,template_x,template_y,template_z,stain,sdt_CD31,index,name,subject,participant_id,lightsheet_id,genotype,sex,treatment
0,4.105234,3.538553,-0.856197,438092.0,0.110419,-6.757585,4.041916,Abeta,0.004723,0,left root,sub-AS40F2,sub-AS40F2,a,ApoE4,F,PBS
1,4.283532,3.562272,-0.850668,437739.0,0.390061,-6.739017,4.070789,Abeta,0.005129,0,left root,sub-AS40F2,sub-AS40F2,a,ApoE4,F,PBS
2,4.278144,3.600416,-0.864738,331.0,0.382313,-6.709099,4.048942,Abeta,0.011406,11046,right Declive (VI),sub-AS40F2,sub-AS40F2,a,ApoE4,F,PBS
3,4.300077,3.604410,-0.874487,875.0,0.416069,-6.709476,4.041669,Abeta,0.002918,11046,right Declive (VI),sub-AS40F2,sub-AS40F2,a,ApoE4,F,PBS
4,2.956877,4.077727,-0.860799,544653.0,-1.594131,-6.193312,3.927016,Abeta,-0.002007,1067,left Simple lobule,sub-AS40F2,sub-AS40F2,a,ApoE4,F,PBS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1317402,1.124415,-19.020298,-6.857144,288.0,-4.474774,-10.561914,-1.761324,Abeta,-0.002620,0,left root,sub-AS208F3,sub-AS208F3,h,ApoE4,F,PBS
1317403,3.052669,-19.268793,-6.689284,707.0,-2.718500,-10.825722,-1.489497,Abeta,0.002846,0,left root,sub-AS208F3,sub-AS208F3,h,ApoE4,F,PBS
1317404,3.700901,-19.332867,-6.561223,706.0,-2.129496,-10.893318,-1.326221,Abeta,-0.012109,0,left root,sub-AS208F3,sub-AS208F3,h,ApoE4,F,PBS
1317405,3.544901,-20.154963,-6.645412,1767.0,-2.292761,-11.657829,-1.367397,Abeta,-0.010910,0,left root,sub-AS208F3,sub-AS208F3,h,ApoE4,F,PBS


In [25]:
#add more derived vars

df["sdt_CD31_um"] = df["sdt_CD31"] * 1000.0

voxel_vol = 1.6 * 1.6 * 2.75  # um^3
voxel_vol_ml = 0.0016 * 0.0016 * 0.00275

df["plaque_vol_um3"] = df["nvoxels"] * voxel_vol
df["plaque_vol_ml"] = df["nvoxels"] * voxel_vol_ml
df["equiv_diam_um"] = 2 * ((3 * df["plaque_vol_um3"]) / (4 * np.pi)) ** (1/3)


In [28]:
df

,pos_x,pos_y,pos_z,nvoxels,template_x,template_y,template_z,stain,sdt_CD31,index,...,subject,participant_id,lightsheet_id,genotype,sex,treatment,sdt_CD31_um,plaque_vol_um3,plaque_vol_ml,equiv_diam_um
0,4.105234,3.538553,-0.856197,438092.0,0.110419,-6.757585,4.041916,Abeta,0.004723,0,...,sub-AS40F2,sub-AS40F2,a,ApoE4,F,PBS,4.722885,3084167.68,0.003084,180.598076
1,4.283532,3.562272,-0.850668,437739.0,0.390061,-6.739017,4.070789,Abeta,0.005129,0,...,sub-AS40F2,sub-AS40F2,a,ApoE4,F,PBS,5.129252,3081682.56,0.003082,180.549557
2,4.278144,3.600416,-0.864738,331.0,0.382313,-6.709099,4.048942,Abeta,0.011406,11046,...,sub-AS40F2,sub-AS40F2,a,ApoE4,F,PBS,11.405924,2330.24,0.000002,16.448792
3,4.300077,3.604410,-0.874487,875.0,0.416069,-6.709476,4.041669,Abeta,0.002918,11046,...,sub-AS40F2,sub-AS40F2,a,ApoE4,F,PBS,2.918425,6160.00,0.000006,22.743678
4,2.956877,4.077727,-0.860799,544653.0,-1.594131,-6.193312,3.927016,Abeta,-0.002007,1067,...,sub-AS40F2,sub-AS40F2,a,ApoE4,F,PBS,-2.006530,3834357.12,0.003834,194.191990
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1317402,1.124415,-19.020298,-6.857144,288.0,-4.474774,-10.561914,-1.761324,Abeta,-0.002620,0,...,sub-AS208F3,sub-AS208F3,h,ApoE4,F,PBS,-2.620266,2027.52,0.000002,15.703225
1317403,3.052669,-19.268793,-6.689284,707.0,-2.718500,-10.825722,-1.489497,Abeta,0.002846,0,...,sub-AS208F3,sub-AS208F3,h,ApoE4,F,PBS,2.846298,4977.28,0.000005,21.183505
1317404,3.700901,-19.332867,-6.561223,706.0,-2.129496,-10.893318,-1.326221,Abeta,-0.012109,0,...,sub-AS208F3,sub-AS208F3,h,ApoE4,F,PBS,-12.109202,4970.24,0.000005,21.173513
1317405,3.544901,-20.154963,-6.645412,1767.0,-2.292761,-11.657829,-1.367397,Abeta,-0.010910,0,...,sub-AS208F3,sub-AS208F3,h,ApoE4,F,PBS,-10.909772,12439.68,0.000012,28.747728


In [27]:
df.to_parquet('data.parquet')